# Pydantic Nested Models: Building Structured Hierarchies

Standard inputs often represent rich hierarchies of structures (e.g. users containing contact lists, containing addresses). Defining these as nested models in Pydantic:
1. Enhances structural validation (each depth level has its own strict schema validation).
2. Supports recursive parsing (standard Python dicts containing dicts automatically convert to model class instances).

In this notebook, we cover:
- Creating sub-models `ContactDetails` and `AddressDetails`.
- Nesting these models within a parent `Patient` model.
- Automatically validating nested structures from compound inputs.


In [ ]:
!uv pip install pydantic 'pydantic[email]' --quiet


## 1. Imports and Sub-Models


In [ ]:
from pydantic import BaseModel, EmailStr, AnyUrl, Field, field_validator, model_validator, computed_field
from typing import List, Dict, Optional, Annotated


In [ ]:
class ContactDetails(BaseModel):
    email_id: Annotated[EmailStr, Field(description='Primary email address of the patient', examples=['john@example.com'])]
    contact_number: Annotated[str, Field(min_length=10, max_length=15, description='Primary contact number', examples=['9999999999'])]
    emergency_contact_number: Annotated[Optional[str], Field(default=None, min_length=10, max_length=15, description='Emergency contact number', examples=['8888888888'])]

    @field_validator('email_id')
    @classmethod
    def email_validator(cls, value: str) -> str:
        valid_domain = ['domain.io', 'example.com']
        domain_name = value.split("@")[-1]

        if domain_name not in valid_domain:
            raise ValueError('Not a valid domain')
        
        return value


## 2. Defining Address Details

We add an `AddressDetails` model to store the patient's address structure.


In [ ]:
class AddressDetails(BaseModel):
    city: str
    state: str
    postalCode: str


## 3. Nesting Sub-Models under Patient

In the `Patient` model below, both `contact_details` and `address_details` are typed using their respective model schemas. When Pydantic parses input dictionaries, it will validate and instantiate these nested models recursively.


In [ ]:
class Patient(BaseModel):

    name: str = Annotated[str, Field(max_length=150, title='Name of the patient', description='Patient Name for records', examples=['John Doe'])]
    age: int
    linkedin_url: Optional[AnyUrl] = None
    weight: Annotated[float, Field(gt=0, description='Submit patient weight for the report', strict=True)]
    height: Annotated[float, Field(gt=0, description='Submit patient height for the report', strict=True)]
    married: Annotated[bool, Field(default=None, description='Is the patient married or not')]
    allergies: Annotated[Optional[List[str]], Field(default=None, max_length=5)]
    
    # Nesting the sub-models
    contact_details: ContactDetails
    address_details: AddressDetails

    @field_validator('name')
    @classmethod
    def transform_name(cls, value: str) -> str:
        return value.upper()
    
    @field_validator('age', mode='before')
    @classmethod
    def validate_age(cls, value: int) -> int:
        if 0 < value < 120:
            return value
        else:
            raise ValueError("Age should be in between 0 and 120")

    @model_validator(mode='after')
    def validate_emergency_contact(self):
        if self.age > 60 and self.contact_details.emergency_contact_number is None:
            raise ValueError('Emergency contact number is mandatory for patients above 60 years of age')
        return self
    
    @computed_field
    @property
    def calculate_bmi(self) -> float:
        bmi = round(self.weight / (self.height**2), 2)
        return bmi


## 4. Data Processing Pipelines


In [ ]:
def insert_patient_data(patient: Patient):
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    print(f"Patient Address Details: {patient.address_details}")
    print(f"Patient Calculated BMI: {patient.calculate_bmi}")
    print('Patient info inserted')


In [ ]:
def update_patient_data(patient: Patient):
    print(f"Patient Name: {patient.name}")
    print(f"Patient Age: {patient.age}")
    print(f"Patient Weight: {patient.weight}")
    print(f"Patient Married Status: {patient.married}")
    print(f"Patient Allergies: {patient.allergies}")
    print(f"Patient Contact Details: {patient.contact_details}")
    print(f"Patient Calculated BMI: {patient.calculate_bmi}")
    print(f"Patient Address Details: {patient.address_details}")
    print('Patient info updateed')


## 5. Simulating Nested Dictionary Payload

We construct separate dictionary payloads representing parts of our patient schema and combine them into a single nested payload.


In [ ]:
address_detail = {
    'city': 'Gurugram',
    'state': 'Haryana',
    'postalCode': '122021'
}


In [ ]:
contact_details = {
    'email_id': 'example@domain.io',
    'contact_number': '9999999999'
}


In [ ]:
patient_info = {
    'name': 'Kevin',
    'age': 26,
    'weight': 67.9,
    'height': 1.73,
    'married': False,
    'allergies': ['lactose', 'dust'],
    'contact_details': contact_details,
    'address_details': address_detail
}


## 6. Model Instantiation

When we instantiate the `Patient` model, Pydantic recursively parses and instantiates both `ContactDetails` and `AddressDetails` instances.


In [ ]:
patient = Patient(**patient_info)
patient


In [ ]:
insert_patient_data(patient=patient)


In [ ]:
update_patient_data(patient=patient)
